<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-04/blob/main/hw_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4 — DML, DDL and Complex SQL Expressions

## 1. Setup and Data Loading

In [31]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [32]:
!pip install -q psycopg2-binary sqlalchemy pandas kagglehub

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw4_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [33]:
# Dataset import
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [34]:
import os

sorted(os.listdir(dataset_dir))

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [35]:
files = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'olist_order_reviews_dataset.csv',
]

for file_name in files:
    df_tmp = pd.read_csv(os.path.join(dataset_dir, file_name))
    print(f"\n{file_name}")
    print("shape:", df_tmp.shape)
    print("columns:", df_tmp.columns.tolist())


olist_customers_dataset.csv
shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_orders_dataset.csv
shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_order_items_dataset.csv
shape: (112650, 7)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_products_dataset.csv
shape: (32951, 9)
columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

olist_sellers_dataset.csv
shape: (3095, 4)
columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

olist_order_reviews_dataset.csv
shape: (99224, 7)
columns

In [36]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
    'olist_order_reviews_raw': 'olist_order_reviews_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)

    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )

    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


### 1.2 Create Typed Tables with Constraints

In [37]:
with engine.begin() as conn:
    conn.execute(text("""

      DROP TABLE IF EXISTS customer_segments    CASCADE;
      DROP TABLE IF EXISTS seller_score         CASCADE;
      DROP TABLE IF EXISTS seller_alerts        CASCADE;
      DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
      DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
      DROP TABLE IF EXISTS olist_order_items    CASCADE;
      DROP TABLE IF EXISTS olist_orders         CASCADE;
      DROP TABLE IF EXISTS olist_customers      CASCADE;
      DROP TABLE IF EXISTS olist_products       CASCADE;
      DROP TABLE IF EXISTS olist_sellers        CASCADE;

      CREATE TABLE olist_customers (
          customer_id              TEXT PRIMARY KEY,
          customer_unique_id       TEXT NOT NULL,
          customer_zip_code_prefix INTEGER,
          customer_city            TEXT,
          customer_state           CHAR(2) NOT NULL
      );

      CREATE INDEX idx_olist_customers_unique_id
          ON olist_customers (customer_unique_id);

      CREATE TABLE olist_orders (
          order_id                      TEXT PRIMARY KEY,
          customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
          order_status                  TEXT NOT NULL,
          order_purchase_timestamp      TIMESTAMP NOT NULL,
          order_approved_at             TIMESTAMP,
          order_delivered_carrier_date  TIMESTAMP,
          order_delivered_customer_date TIMESTAMP,
          order_estimated_delivery_date TIMESTAMP,
          CHECK (order_status IN (
              'created', 'approved', 'invoiced', 'processing',
              'shipped', 'delivered', 'unavailable', 'canceled'
          ))
      );

      CREATE TABLE olist_products (
          product_id                 TEXT PRIMARY KEY,
          product_category_name      TEXT,
          product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
          product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
          product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
          product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
          product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
          product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
          product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
      );

      CREATE TABLE olist_sellers (
          seller_id              TEXT PRIMARY KEY,
          seller_zip_code_prefix INTEGER,
          seller_city            TEXT,
          seller_state           CHAR(2) NOT NULL
      );

      CREATE TABLE olist_order_items (
          order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
          order_item_id       INTEGER NOT NULL,
          product_id          TEXT REFERENCES olist_products(product_id),
          seller_id           TEXT REFERENCES olist_sellers(seller_id),
          shipping_limit_date TIMESTAMP,
          price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
          freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
          PRIMARY KEY (order_id, order_item_id)
      );

      CREATE INDEX idx_olist_order_items_seller_order
          ON olist_order_items (seller_id, order_id);

      CREATE TABLE olist_order_reviews (
          review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
          review_id                TEXT,
          order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
          review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
          review_comment_title     TEXT,
          review_comment_message   TEXT,
          review_creation_date     TIMESTAMP,
          review_answer_timestamp  TIMESTAMP
      );

      CREATE INDEX idx_olist_order_reviews_order_id
          ON olist_order_reviews (order_id);

    """))
print("Successfully created typed tables.")

Successfully created typed tables.


### 1.3 Load Data from Raw to Typed Tables

In [38]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO olist_customers
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix::INTEGER,
            customer_city,
            customer_state::CHAR(2)
        FROM olist_customers_raw;

        INSERT INTO olist_orders
        SELECT
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::TIMESTAMP,
            order_approved_at::TIMESTAMP,
            order_delivered_carrier_date::TIMESTAMP,
            order_delivered_customer_date::TIMESTAMP,
            order_estimated_delivery_date::TIMESTAMP
        FROM olist_orders_raw;

        INSERT INTO olist_products
        SELECT
            product_id,
            product_category_name,
            product_name_lenght::INTEGER,
            product_description_lenght::INTEGER,
            product_photos_qty::INTEGER,
            product_weight_g::INTEGER,
            product_length_cm::INTEGER,
            product_height_cm::INTEGER,
            product_width_cm::INTEGER
        FROM olist_products_raw;

        INSERT INTO olist_sellers
        SELECT
            seller_id,
            seller_zip_code_prefix::INTEGER,
            seller_city,
            seller_state::CHAR(2)
        FROM olist_sellers_raw;

        INSERT INTO olist_order_items
        SELECT
            order_id,
            order_item_id::INTEGER,
            product_id,
            seller_id,
            shipping_limit_date::TIMESTAMP,
            price::NUMERIC(12, 2),
            freight_value::NUMERIC(12, 2)
        FROM olist_order_items_raw;

        INSERT INTO olist_order_reviews (
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            review_creation_date,
            review_answer_timestamp
        )
        SELECT
            review_id,
            order_id,
            review_score::SMALLINT,
            review_comment_title,
            review_comment_message,
            review_creation_date::TIMESTAMP,
            review_answer_timestamp::TIMESTAMP
        FROM olist_order_reviews_raw;
    """))

print("Raw data successfully loaded into typed tables.")

Raw data successfully loaded into typed tables.


### 1.4 Validate Typed Table Row Counts

In [39]:
pd.read_sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS n_rows
    FROM olist_customers

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM olist_orders

    UNION ALL

    SELECT 'order_items', COUNT(*)
    FROM olist_order_items

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM olist_products

    UNION ALL

    SELECT 'sellers', COUNT(*)
    FROM olist_sellers

    UNION ALL

    SELECT 'reviews', COUNT(*)
    FROM olist_order_reviews

    ORDER BY table_name;
""", engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


All six typed tables contain the same number of rows as their corresponding raw tables. No records were lost during type conversion and loading.

## 2. DML queries with PostgreSQL features

### 2.1 INSERT ... RETURNING

In [40]:
with engine.begin() as conn:
    # Create the alerts table
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_alerts CASCADE;

        CREATE TABLE seller_alerts (
            alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
            alert_type TEXT NOT NULL
                CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
            severity   SMALLINT NOT NULL
                CHECK (severity BETWEEN 1 AND 5),
            details    TEXT,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    # Find the seller with the most late deliveries and create an alert
    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),
        late_sellers AS (
            SELECT
                so.seller_id,
                COUNT(DISTINCT o.order_id) AS late_orders
            FROM seller_orders AS so
            JOIN olist_orders AS o
                ON o.order_id = so.order_id
            WHERE o.order_delivered_customer_date IS NOT NULL
              AND o.order_estimated_delivery_date IS NOT NULL
              AND o.order_delivered_customer_date::DATE
                    > o.order_estimated_delivery_date::DATE
            GROUP BY so.seller_id
            ORDER BY late_orders DESC, so.seller_id
            LIMIT 1
        )
        INSERT INTO seller_alerts (
            seller_id,
            alert_type,
            severity,
            details
        )
        SELECT
            seller_id,
            'late_delivery',
            4,
            'Seller has ' || late_orders ||
            ' late delivered orders in the loaded dataset'
        FROM late_sellers
        RETURNING
            alert_id,
            seller_id,
            alert_type,
            severity,
            details,
            created_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)

,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-09-29 15:41:04.302684+00:00


The query created a late_delivery alert for the seller with the highest number of late delivered orders. The seller had 172 late deliveries. RETURNING exposed the inserted row immediately, including the generated alert_id and timestamp, which makes the operation easy to audit in the notebook.

### 2.2 INSERT ... ON CONFLICT DO UPDATE

In [41]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS seller_score CASCADE;

        CREATE TABLE seller_score (
            seller_id   TEXT PRIMARY KEY
                REFERENCES olist_sellers(seller_id),
            avg_rating  NUMERIC(3, 2),
            n_reviews   INTEGER NOT NULL DEFAULT 0
                CHECK (n_reviews >= 0),
            tier        TEXT
                CHECK (tier IN ('gold', 'silver', 'bronze')),
            updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
        );
    """))

    result = conn.execute(text("""
        WITH seller_orders AS (
            SELECT DISTINCT
                oi.seller_id,
                oi.order_id
            FROM olist_order_items AS oi
        ),

        review_per_order AS (
            SELECT
                rv.order_id,
                AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
            FROM olist_order_reviews AS rv
            WHERE rv.review_score IS NOT NULL
            GROUP BY rv.order_id
        ),

        seller_rating AS (
            SELECT
                so.seller_id,
                AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
                COUNT(rpo.order_id) AS n_reviews
            FROM seller_orders AS so
            LEFT JOIN review_per_order AS rpo
                ON rpo.order_id = so.order_id
            GROUP BY so.seller_id
        )

        INSERT INTO seller_score (
            seller_id,
            avg_rating,
            n_reviews,
            tier
        )
        SELECT
            seller_id,
            avg_rating,
            n_reviews,
            CASE
                WHEN avg_rating IS NULL THEN NULL
                WHEN avg_rating >= 4.5 THEN 'gold'
                WHEN avg_rating >= 3.5 THEN 'silver'
                ELSE 'bronze'
            END AS tier
        FROM seller_rating

        ON CONFLICT (seller_id) DO UPDATE
        SET
            avg_rating = EXCLUDED.avg_rating,
            n_reviews  = EXCLUDED.n_reviews,
            tier       = EXCLUDED.tier,
            updated_at = NOW()

        RETURNING
            seller_id,
            avg_rating,
            n_reviews,
            tier,
            updated_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

seller_score_df = pd.DataFrame(rows, columns=columns)

seller_score_df.head(10)

,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-09-29 15:41:04.461640+00:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-09-29 15:41:04.461640+00:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-09-29 15:41:04.461640+00:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-09-29 15:41:04.461640+00:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-09-29 15:41:04.461640+00:00
5,fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-09-29 15:41:04.461640+00:00
6,5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-09-29 15:41:04.461640+00:00
7,b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-09-29 15:41:04.461640+00:00
8,ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-09-29 15:41:04.461640+00:00
9,a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-09-29 15:41:04.461640+00:00


In [42]:
# Disrtibution by tier
with engine.begin() as conn:
    result = conn.execute(text("""
      SELECT
          COALESCE(tier, 'no_rating') AS tier_group,
          COUNT(*) AS seller_count
      FROM seller_score
      GROUP BY tier
      ORDER BY seller_count DESC;
    """))
    rows = result.fetchall()
    columns = result.keys()

pd.DataFrame(rows, columns=columns)


,tier_group,seller_count
0,silver,1568
1,gold,983
2,bronze,539
3,no_rating,5


Most sellers fall into the silver tier, followed by gold and bronze. Only 5 sellers have no rating, which indicates that review coverage is high across the seller population.

### 2.3 UPDATE ... FROM

In [43]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE olist_orders
        ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;
    """))

    result = conn.execute(text("""
        WITH delivery_flags AS (
            SELECT
                order_id,
                (
                    order_delivered_customer_date IS NOT NULL
                    AND order_estimated_delivery_date IS NOT NULL
                    AND order_delivered_customer_date::DATE
                        > order_estimated_delivery_date::DATE
                ) AS is_late_calc
            FROM olist_orders
        )

        UPDATE olist_orders AS o
        SET is_late = f.is_late_calc
        FROM delivery_flags AS f
        WHERE f.order_id = o.order_id
        RETURNING
            o.order_id,
            o.customer_id,
            o.is_late;
    """))

    rows = result.fetchall()
    columns = result.keys()

updated_orders_df = pd.DataFrame(rows, columns=columns)
updated_orders_df.head(10)

,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


In [44]:
pd.read_sql("""
    SELECT
        is_late,
        COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY is_late
    ORDER BY is_late;
""", engine)

,is_late,n_orders
0,False,92906
1,True,6535


The late-delivery flag was recalculated for all orders from the delivery dates. Because the value is derived rather than incrementally modified, rerunning the update produces the same business state for unchanged source data.

### 2.4 DELETE ... RETURNING — Remove Old Seller Alerts

In [45]:
with engine.begin() as conn:
    result = conn.execute(text("""
        DELETE FROM seller_alerts
        WHERE created_at < NOW() - INTERVAL '30 days'
        RETURNING
            alert_id,
            seller_id,
            alert_type,
            created_at;
    """))

    rows = result.fetchall()
    columns = result.keys()

deleted_alerts_df = pd.DataFrame(rows, columns=columns)
deleted_alerts_df

,alert_id,seller_id,alert_type,created_at


No alerts were deleted because all current alerts were created recently and are less than 30 days old. An empty result is therefore expected in this dataset state.

## 3. Customer Segmentation

### 3.1 Create customer dimension

In [46]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS customer_segments CASCADE;
        DROP TABLE IF EXISTS olist_customer_dim CASCADE;

        CREATE TABLE olist_customer_dim (
            customer_unique_id TEXT PRIMARY KEY,
            latest_customer_id TEXT REFERENCES olist_customers(customer_id),
            customer_state     CHAR(2),
            customer_city      TEXT
        );

        INSERT INTO olist_customer_dim (
            customer_unique_id,
            latest_customer_id,
            customer_state,
            customer_city
        )
        SELECT DISTINCT ON (c.customer_unique_id)
            c.customer_unique_id,
            c.customer_id AS latest_customer_id,
            c.customer_state,
            c.customer_city
        FROM olist_customers AS c
        LEFT JOIN olist_orders AS o
            ON o.customer_id = c.customer_id
        ORDER BY
            c.customer_unique_id,
            o.order_purchase_timestamp DESC NULLS LAST,
            c.customer_id;
    """))

### 3.2 Create and populate customer_segments

In [47]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE customer_segments (
            segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

            customer_unique_id  TEXT NOT NULL,

            segment_name        TEXT NOT NULL
                CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

            rfm_score           SMALLINT NOT NULL
                CHECK (rfm_score BETWEEN 1 AND 5),

            recency_days        INTEGER
                CHECK (recency_days IS NULL OR recency_days >= 0),

            monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
                CHECK (monetary_value >= 0),

            n_orders            INTEGER NOT NULL DEFAULT 0
                CHECK (n_orders >= 0),

            assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

            UNIQUE (customer_unique_id)
        );

        ALTER TABLE customer_segments
            ADD CONSTRAINT fk_customer_segments_customer_unique
            FOREIGN KEY (customer_unique_id)
            REFERENCES olist_customer_dim(customer_unique_id)
            ON DELETE CASCADE;
    """))

    result = conn.execute(text("""
        WITH reference_date AS (
            SELECT
                (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
            FROM olist_orders
        ),

        order_totals AS (
            SELECT
                o.order_id,
                o.customer_id,
                o.order_purchase_timestamp::DATE AS order_date,
                COALESCE(
                    SUM(oi.price + oi.freight_value),
                    0
                )::NUMERIC(12, 2) AS order_total
            FROM olist_orders AS o
            LEFT JOIN olist_order_items AS oi
                ON oi.order_id = o.order_id
            GROUP BY
                o.order_id,
                o.customer_id,
                o.order_purchase_timestamp
        ),

        customer_stats AS (
            SELECT
                c.customer_unique_id,
                COUNT(DISTINCT ot.order_id) AS n_orders,
                COALESCE(
                    SUM(ot.order_total),
                    0
                )::NUMERIC(12, 2) AS monetary_value,
                MAX(ot.order_date) AS last_order_date
            FROM olist_customer_dim AS cd
            JOIN olist_customers AS c
                ON c.customer_unique_id = cd.customer_unique_id
            LEFT JOIN order_totals AS ot
                ON ot.customer_id = c.customer_id
            GROUP BY c.customer_unique_id
        ),

        segmented AS (
            SELECT
                cs.customer_unique_id,
                cs.n_orders,
                cs.monetary_value,

                CASE
                    WHEN cs.last_order_date IS NULL THEN NULL
                    ELSE (
                        rd.as_of_date - cs.last_order_date
                    )::INTEGER
                END AS recency_days,

                CASE
                    WHEN cs.monetary_value >= 1000 THEN 'VIP'

                    WHEN cs.last_order_date IS NOT NULL
                         AND (
                             rd.as_of_date - cs.last_order_date
                         )::INTEGER <= 90
                         AND cs.n_orders = 1
                    THEN 'new'

                    WHEN cs.n_orders >= 2
                         OR cs.monetary_value >= 100
                    THEN 'regular'

                    ELSE 'inactive'
                END AS segment_name,

                CASE
                    WHEN cs.monetary_value >= 1000 THEN 5
                    WHEN cs.monetary_value >= 500  THEN 4
                    WHEN cs.n_orders >= 2          THEN 3
                    WHEN cs.n_orders = 1           THEN 2
                    ELSE 1
                END::SMALLINT AS rfm_score

            FROM customer_stats AS cs
            CROSS JOIN reference_date AS rd
        )

        INSERT INTO customer_segments (
            customer_unique_id,
            segment_name,
            rfm_score,
            recency_days,
            monetary_value,
            n_orders
        )

        SELECT
            customer_unique_id,
            segment_name,
            rfm_score,
            recency_days,
            monetary_value,
            n_orders
        FROM segmented

        ON CONFLICT (customer_unique_id) DO UPDATE
        SET
            segment_name   = EXCLUDED.segment_name,
            rfm_score      = EXCLUDED.rfm_score,
            recency_days   = EXCLUDED.recency_days,
            monetary_value = EXCLUDED.monetary_value,
            n_orders       = EXCLUDED.n_orders,
            assigned_at    = NOW();
    """))

In [48]:
pd.read_sql("""
    SELECT
        segment_name,
        COUNT(*) AS n_customers,
        ROUND(AVG(monetary_value), 2) AS avg_monetary_value,
        ROUND(AVG(n_orders), 2) AS avg_orders
    FROM customer_segments
    GROUP BY segment_name
    ORDER BY n_customers DESC;
""", engine)

,segment_name,n_customers,avg_monetary_value,avg_orders
0,regular,45982,224.57,1.07
1,inactive,40032,59.28,1.00
2,new,8891,140.39,1.00
3,VIP,1191,1591.81,1.11


In [49]:
pd.read_sql("""
    SELECT
        (SELECT COUNT(*) FROM olist_customer_dim) AS customer_dim_rows,
        (SELECT COUNT(*) FROM customer_segments) AS segment_rows;
""", engine)

,customer_dim_rows,segment_rows
0,96096,96096


Customer segmentation was successfully created for all 96,096 unique customers. The largest groups are regular and inactive, while VIP customers represent a much smaller segment with substantially higher average monetary value. The equal row counts in olist_customer_dim and customer_segments confirm one segmentation record per real customer.

## Task 4 — JOIN Queries

### 4.1 Which product categories and seller states generate the highest revenue from delivered orders

In [50]:
pd.read_sql("""
    SELECT
        p.product_category_name,
        s.seller_state,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    JOIN olist_sellers AS s
        ON s.seller_id = oi.seller_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        p.product_category_name,
        s.seller_state
    ORDER BY revenue DESC NULLS LAST
    LIMIT 10;
""", engine)

,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


The highest revenue combinations are concentrated in sellers from SP. cama_mesa_banho and relogios_presentes generate the largest delivered-order revenue among the top results

### 4.2 What customer-level features can be built for each real customer

In [51]:
pd.read_sql("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            c.customer_unique_id,
            SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
        LEFT JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            c.customer_unique_id
    ),

    review_per_order AS (
        SELECT
            rv.order_id,
            AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
        FROM olist_order_reviews AS rv
        WHERE rv.review_score IS NOT NULL
        GROUP BY rv.order_id
    ),

    customer_stats AS (
        SELECT
            ot.customer_unique_id,
            COUNT(DISTINCT ot.order_id) AS n_orders,
            COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
            AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
        FROM order_totals AS ot
        LEFT JOIN review_per_order AS rpo
            ON rpo.order_id = ot.order_id
        GROUP BY ot.customer_unique_id
    )

    SELECT
        cd.customer_unique_id,
        cd.customer_state,
        COALESCE(cs.n_orders, 0) AS n_orders,
        COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
        COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
    FROM olist_customer_dim AS cd
    LEFT JOIN customer_stats AS cs
        ON cs.customer_unique_id = cd.customer_unique_id
    ORDER BY total_spend DESC NULLS LAST
    LIMIT 20;
""", engine)

,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.0
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.0
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.0
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.0
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.0
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.0


Customer-level features were successfully created at the customer_unique_id level. The output combines order count, total spend, and average review score while preserving customers even when some related data is missing.

### 4.3 Which states appear among customers, sellers, or both

In [52]:
pd.read_sql("""
    WITH customer_states AS (
        SELECT
            customer_state AS state,
            COUNT(*) AS n_customers
        FROM olist_customers
        GROUP BY customer_state
    ),

    seller_states AS (
        SELECT
            seller_state AS state,
            COUNT(*) AS n_sellers
        FROM olist_sellers
        GROUP BY seller_state
    )

    SELECT
        COALESCE(c.state, s.state) AS state,
        c.n_customers,
        s.n_sellers,
        CASE
            WHEN c.state IS NOT NULL
             AND s.state IS NOT NULL THEN 'both'
            WHEN c.state IS NOT NULL THEN 'customers_only'
            ELSE 'sellers_only'
        END AS state_presence
    FROM customer_states AS c
    FULL OUTER JOIN seller_states AS s
        ON s.state = c.state
    ORDER BY state;
""", engine)

,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


Most Brazilian states in the result contain both customers and sellers, but some states such as AL and AP appear only on the customer side. The FULL OUTER JOIN makes these one-sided cases visible.

### 4.4 Which seller pairs are located in the same state but different cities

In [53]:
pd.read_sql("""
    SELECT
        s1.seller_id AS seller_a,
        s2.seller_id AS seller_b,
        s1.seller_state,
        s1.seller_city AS city_a,
        s2.seller_city AS city_b
    FROM olist_sellers AS s1
    JOIN olist_sellers AS s2
        ON s1.seller_state = s2.seller_state
       AND s1.seller_id < s2.seller_id
       AND s1.seller_city IS DISTINCT FROM s2.seller_city
    ORDER BY
        s1.seller_state,
        seller_a,
        seller_b
    LIMIT 20;
""", engine)

,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


The query identifies pairs of different sellers located in the same state but in different cities. The first returned pairs are from BA because the result is ordered by state and limited to 20 rows.

### 4.5 Which orders have no review record

In [54]:
pd.read_sql("""
    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp
    FROM olist_orders AS o
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = o.order_id
    WHERE rv.review_row_id IS NULL
    ORDER BY o.order_purchase_timestamp
    LIMIT 20;
""", engine)

,order_id,customer_id,order_status,order_purchase_timestamp
0,cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
1,5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
2,02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
3,0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
4,c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
5,d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
6,05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
7,719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
8,fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
9,171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


The query finds orders that do not have any matching review record. The result includes orders with different statuses, including delivered orders, showing that some completed orders still have no customer review.

## Task 5- GROUP BY, HAVING, aggregates

### 5.1 Which product categories generate significant revenue

In [55]:
pd.read_sql("""
    SELECT
        p.product_category_name,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
        SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
        AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
    FROM olist_order_items AS oi
    JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY p.product_category_name
    HAVING SUM(oi.price) > 10000
    ORDER BY gross_revenue DESC NULLS LAST;
""", engine)

,product_category_name,n_orders,gross_revenue,freight,avg_item_price
0,beleza_saude,8836,1258681.34,182566.73,130.16
1,relogios_presentes,5624,1205005.68,100535.93,201.14
2,cama_mesa_banho,9417,1036988.68,204693.04,93.30
3,esporte_lazer,7720,988048.97,168607.51,114.34
4,informatica_acessorios,6689,911954.32,147318.08,116.51
5,moveis_decoracao,6449,729762.49,172749.30,87.56
6,cool_stuff,3632,635290.85,84039.10,167.36
7,utilidades_domesticas,5884,632248.66,146149.11,90.79
8,automotivo,3897,592720.11,92664.21,139.96
9,ferramentas_jardim,3518,485256.46,98962.75,111.63


beleza_saude has the highest gross revenue at about 1.26M, followed by relogios_presentes and cama_mesa_banho. Some high-revenue categories differ substantially in average item price; for example, pcs has relatively few orders but an average item price above 1,000. The presence of a NULL product category also indicates a data-quality issue in the product metadata.

### 5.2 Which customer states have the highest average order value among states with sufficient order volume

In [56]:
pd.read_sql("""
    WITH order_totals AS (
        SELECT
            o.order_id,
            c.customer_state,
            SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
        JOIN olist_order_items AS oi
            ON oi.order_id = o.order_id
        GROUP BY
            o.order_id,
            c.customer_state
    )

    SELECT
        customer_state,
        COUNT(*) AS n_orders,
        SUM(order_total)::NUMERIC(12, 2) AS gmv,
        (
            SUM(order_total) / NULLIF(COUNT(*), 0)
        )::NUMERIC(8, 2) AS aov
    FROM order_totals
    GROUP BY customer_state
    HAVING COUNT(*) >= 100
    ORDER BY aov DESC NULLS LAST;
""", engine)

,customer_state,n_orders,gmv,aov
0,PB,532,140987.81,265.01
1,AL,411,96229.40,234.13
2,RO,247,57558.02,233.03
3,PA,970,217647.11,224.38
4,TO,279,61354.42,219.91
5,PI,493,108132.28,219.34
6,SE,345,73032.32,211.69
7,RN,482,101895.08,211.40
8,CE,1327,275606.30,207.69
9,MT,903,186168.96,206.17


PB has the highest average order value at 265.01 among states with at least 100 orders, followed by AL and RO. SP has by far the largest order volume and GMV, but its average order value is lower at 143.12. This shows that high total revenue does not necessarily imply high average order value.

### 5.3 Which product categories were purchased by each real customer

In [57]:
pd.read_sql("""
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS n_orders,
        SUM(oi.price)::NUMERIC(12, 2) AS total_spend,

        STRING_AGG(
            DISTINCT COALESCE(
                p.product_category_name,
                '(unknown)'
            ),
            ', ' ORDER BY COALESCE(
                p.product_category_name,
                '(unknown)'
            )
        ) AS categories

    FROM olist_customers AS c
    JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id

    GROUP BY c.customer_unique_id

    HAVING COUNT(DISTINCT o.order_id) >= 1

    ORDER BY total_spend DESC NULLS LAST
    LIMIT 20;
""", engine)

,customer_unique_id,n_orders,total_spend,categories
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,telefonia_fixa
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,telefonia_fixa
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,utilidades_domesticas
4,459bef486812aa25204be022145caa62,1,6729.00,pcs
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,artes
6,4007669dec559734d6f53e029e360987,1,5934.60,agro_industria_e_comercio
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,eletroportateis
8,5d0a2980b292d049061542014e8960bf,1,4599.90,cool_stuff
9,48e1ac109decbb87765a3eade6854098,1,4590.00,pcs


The highest-spending customers are mostly associated with a small number of expensive product categories, and many of the top customers have only one order. STRING_AGG combines all distinct product categories purchased by each real customer into a single customer-level feature. One top customer has (unknown) as the category, confirming that some product category values are missing.

## Task 6.Set Operation + Window Function

### 6.1 Which states contain both customers and sellers

In [58]:
pd.read_sql("""
    SELECT customer_state AS state
    FROM olist_customers

    INTERSECT

    SELECT seller_state AS state
    FROM olist_sellers

    ORDER BY state;
""", engine)

,state
0,AC
1,AM
2,BA
3,CE
4,DF
5,ES
6,GO
7,MA
8,MG
9,MS


INTERSECT returns only state codes that exist in both the customer and seller datasets. It is appropriate here because the business question asks for the common members of two sets rather than all values from either side.

6.2 Latest Order per Real Customer

In [59]:
pd.read_sql("""
    WITH ranked_orders AS (
        SELECT
            c.customer_unique_id,
            o.customer_id,
            o.order_id,
            o.order_purchase_timestamp,

            ROW_NUMBER() OVER (
                PARTITION BY c.customer_unique_id
                ORDER BY
                    o.order_purchase_timestamp DESC,
                    o.order_id DESC
            ) AS rn

        FROM olist_orders AS o
        JOIN olist_customers AS c
            ON c.customer_id = o.customer_id
    )

    SELECT
        customer_unique_id,
        customer_id,
        order_id,
        order_purchase_timestamp
    FROM ranked_orders
    WHERE rn = 1
    ORDER BY customer_unique_id
    LIMIT 20;
""", engine)

,customer_unique_id,customer_id,order_id,order_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
5,0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
6,00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
7,00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
8,0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
9,0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


## Reflection

## Reflection

The most unusual JOIN for me was the SELF JOIN because the same table is used in two logical roles and additional conditions are needed to avoid duplicate seller pairs. The anti-join was also useful because it provides a simple way to identify missing relationships, such as orders without reviews.

The most difficult queries to formulate as business questions were those focused on technical relationships rather than direct business metrics. Among the PostgreSQL features used in this homework, `ON CONFLICT` seems especially useful for production ETL because it allows repeatable upserts without creating duplicate records. Constraints are also important because they prevent invalid data from entering the typed schema.

In a real ML pipeline, I would use window functions for tasks such as selecting the latest customer event, ranking transactions, generating historical features, and calculating features over time while preserving row-level granularity.

The Olist data also contains several data-quality issues. Some orders have no reviews, some products have missing category values, and review coverage is not complete. These cases should be handled explicitly rather than silently removed.

For customer-level ML features, `customer_unique_id` is more appropriate than `customer_id` because the same real customer may have several order-specific customer IDs. Similarly, before calculating `seller_score`, the data must be reduced to `seller_id × order_id` granularity to prevent one review from being counted multiple times through item-level joins.